# PS2: RL-Enhanced SLM Reasoning — Kaggle Entry Point

This notebook is the Kaggle-specific entry point. It handles:
- Environment setup (P100 GPU constraints)
- W&B authentication
- Sequential execution of the full pipeline

**Run order:** Setup → Baseline → SFT → GRPO → Evaluate → Ablation Table

**HALLUCINATION NOTES in this notebook:**
- Kaggle P100 has 16GB VRAM. Phi-3-mini (3.8B) in float16 + LoRA fits, but barely.
  If you hit OOM: reduce `per_device_train_batch_size` to 1 and `gradient_accumulation_steps` to 4.
- `bfloat16` is NOT supported on P100. Set `torch_dtype: float16` in configs.
- `flash_attention_2` is NOT supported on P100. Use `attn_implementation: eager`.
- Kaggle GPU sessions are ~30h/week. SFT ~10h + GRPO ~18h = tight. Use `limit` args to test first.

In [ ]:
# Cell 1: Install dependencies
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-2000:])
    else:
        print(result.stdout[-500:])

run('pip install trl==1.4.0 peft==0.19.1 lm_eval==0.4.3 wandb pyyaml bitsandbytes -q')

In [ ]:
# Cell 2: GPU check
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
    print('bfloat16 supported:', torch.cuda.is_bf16_supported())
    # On P100: bfloat16=False → use float16 in configs

In [ ]:
# Cell 3: W&B login
import wandb
# Set your W&B API key in Kaggle Secrets: Settings → Add-ons → Secrets
from kaggle_secrets import UserSecretsClient
try:
    secrets = UserSecretsClient()
    wandb_key = secrets.get_secret('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print('W&B logged in')
except Exception as e:
    print(f'W&B login failed (ok for dry runs): {e}')
    wandb.init(mode='disabled')

In [ ]:
# Cell 4: Clone/copy project code
# If running from Kaggle Dataset, adjust path accordingly
import sys, os
PROJECT_DIR = '/kaggle/working/ps2_slm_rl'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# Cell 5: Record baseline (Week 1)
# Use limit=200 for quick check, limit=500 for full baseline
!python scripts/run_baseline.py --limit 200 --batch-size 2

In [ ]:
# Cell 6: Dry run — verify data pipeline
!python scripts/train_sft.py --config configs/sft.yaml --dry-run

In [ ]:
# Cell 7: SFT training (Week 2)
# NOTE: On P100, edit configs/sft.yaml first:
#   torch_dtype: float16
#   attn_implementation: eager
#   per_device_train_batch_size: 1
#   gradient_accumulation_steps: 16
!python scripts/train_sft.py --config configs/sft.yaml

In [ ]:
# Cell 8: GRPO training with curriculum (Week 3-4)
!python scripts/train_grpo.py \
    --config configs/grpo.yaml \
    --curriculum-config configs/curriculum.yaml

In [ ]:
# Cell 9: Merge LoRA for clean inference
!python scripts/merge_lora.py \
    --base-model microsoft/Phi-3-mini-4k-instruct \
    --lora-path checkpoints/curriculum_grpo \
    --output-path final_model \
    --dtype float16

In [ ]:
# Cell 10: Full evaluation + ablation table (Week 5)
!python scripts/evaluate.py \
    --ablation \
    --tasks gsm8k mmlu strategy_qa \
    --batch-size 4 \
    --limit 500

In [ ]:
# Cell 11: Latency benchmark
!python scripts/benchmark_latency.py --model-path ./final_model

In [ ]:
# Cell 12: Print ablation table
import pandas as pd, json
df = pd.read_csv('results/ablation_table.csv', index_col=0)
print(df.to_string())

In [ ]:
# Cell 13: Interactive inference demo
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = './final_model'
dtype = torch.float16  # float16 on P100

tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path, torch_dtype=dtype, device_map='auto'
)
model.eval()

def solve(question: str, max_new_tokens: int = 300) -> str:
    prompt = (
        f'<|system|>\nSolve step by step. End with: #### <number><|end|>\n'
        f'<|user|>\n{question}<|end|>\n'
        f'<|assistant|>\n'
    )
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    generated = output[0][inputs['input_ids'].shape[1]:]
    return tok.decode(generated, skip_special_tokens=True)

# Test
q = "Janet's ducks lay 16 eggs per day. She eats 3 for breakfast every morning and bakes muffins for her friends every day with 4. She sells the remainder at the farmers' market daily for $2 per egg. How much in dollars does she make every day at the farmers' market?"
print('Question:', q)
print('Answer:', solve(q))